# Study report: reproducibility

The study checks that the data of the benchmark are reproducible. It has
three parts: does the same image give the same histograms on another
host, does a rebuild from the sources of the run give the same image,
and does a new safety revision give the old data of the same code
placement? The study has no experiment of its own. For the first two
parts it runs points of other experiments again (mode `rerun`). For the
third part it reads the data of `wake-code-place-q4` (mode `reuse`).

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-measurement-validity@nucleo_g474re/st-reproducibility"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts"))
from probatrix.benchmarks import data  # noqa: E402
from probatrix.benchmarks import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

In [ ]:
import plotly.graph_objects as go
from probatrix.benchmarks.studylib import stats  # noqa: E402

## The data of this instance

One row for each import of `study.yaml`. A `rerun` import needs a run of
this study; a `reuse` import reads the run of the owner study.

In [ ]:
import json

import yaml

design = yaml.safe_load((inst.design_folder / "study.yaml").read_text())
own_folder = {}
state = []
for imp in design.get("imports", []):
    e = imp["experiment"]
    own = inst.folder / f"{data.layout.EXPERIMENT_PREFIX}{e}"
    has_own = (own / "experiment.json").is_file()
    try:
        inst.experiment_folder(e)
        has_any = True
    except FileNotFoundError:
        has_any = False
    if has_own:
        own_folder[e] = own
    state.append({"study": imp["study"], "experiment": e, "mode": imp["mode"], "run of this study": has_own,
                  "run of the owner study": has_any and not has_own,
                  "data for this study": has_own if imp["mode"] == "rerun" else has_any})
pd.DataFrame(state)

## Re-runs on another host (central result same-image-same-data)

For each point of a re-run, the table compares the histograms with the
same point in the run of the owner study, bin for bin. A person confirms
the central result `rebuild-same-image`, because `run.py --rebuild`
writes no record yet.

In [ ]:
same = []
for e, folder in own_folder.items():
    if not any(s["experiment"] == e and s["mode"] == "rerun" for s in state):
        continue
    owners = [f for f in data.layout.instance_folders(inst.root) if f.resolve() != inst.folder
              and (f / f"{data.layout.EXPERIMENT_PREFIX}{e}" / "experiment.json").is_file()]
    if not owners:
        print(f"{e}: no run of the owner study in the results tree")
        continue
    theirs = {(t, p): rep for _, t, p, rep in data.load_instance(owners[0]).points(e)}
    for _, t, p, rep in inst.points(e):
        ref = theirs.get((t, p))
        for m in sorted(rep["hists"]):
            same.append({"experiment": e, "tree": t, "point": p, "metric": m,
                         "the same, bin for bin": None if ref is None or m not in ref["hists"]
                         else stats.hist_identical(rep["hists"][m], ref["hists"][m])})
if same:
    display(pd.DataFrame(same))
else:
    print("No re-run in this instance: the campaign has no run of the rerun imports, so the central result "
          "same-image-same-data has no value.")

## H1: a new safety revision and the old flash data

The flash points `f0` and `f4` of `wake-code-place-q4` have the same code
layout as the points `p0` and `p4` of `wake-code-layout` (commit
`48f1341` of `LATBENCH_CODE_PLACE`). The reference is the run
`wake-code-layout/20261029T002905Z`. The checks have the option
`revision: different`: they give a verdict only when `wake-code-place-q4`
has a run of another safety revision than the reference. The first
table shows the runs: with the same `safety_rev`, H1 stays "not
evaluated". The figure shows the differences against the
limits of the checks, also when the checks give no verdict.

In [ ]:
REF = "wake-code-layout/20261029T002905Z"
MAP = {"f0-default": "p0-default", "f4-default": "p4-default", "f0-noslice": "p0-noslice", "f4-noslice": "p4-noslice"}


def run_json(run_id):
    exp, utc = run_id.split("/")
    for f in data.layout.instance_folders(inst.root):
        p = f / f"{data.layout.EXPERIMENT_PREFIX}{exp}" / f"{utc}.run.json"
        if p.is_file():
            return json.loads(p.read_text())["run"]
    return None


runs = []
if nbfig.built(inst, "wake-code-place-q4"):
    runs = [r["id"] for r in inst.experiment_json("wake-code-place-q4")["runs"] if r["role"] == "evidence"]
info = [dict(run=r, **{k: (run_json(r) or {}).get(k) for k in ("host", "safety_rev", "started")}) for r in runs + [REF]]
pd.DataFrame(info)

In [ ]:
try:
    ref = {x["point"]: x["reps"][0] for x in inst.app_data(REF)["points"] if x["reps"]}
except FileNotFoundError as err:
    print(err)
    ref = {}
diff = []
if ref and nbfig.built(inst, "wake-code-place-q4"):
    new = {p: rep for _, _, p, rep in inst.points("wake-code-place-q4")}
    for cid, metric, st, lim, pts in (("H1 a", "irq", "p50", 1, list(MAP)), ("H1 b", "isr2thr", "p50", 5, list(MAP)),
                                      ("H1 c", None, "drift_per_wake", 0.05, ["f0-default", "f4-default"])):
        for p in pts:
            if p in new and MAP[p] in ref:
                a, b = stats.value(new[p], metric, st), stats.value(ref[MAP[p]], metric, st)
                diff.append({"check": f"{cid}: {metric or 'drift'} {st}", "point": f"{p} - {MAP[p]}", "series": p,
                             "new": a, "reference": b, "value": a - b, "lo": -lim, "hi": lim})
diff = pd.DataFrame(diff)
if len(diff):
    for cid, unit in (("H1 a", "cycles"), ("H1 b", "cycles"), ("H1 c", "cycles per wake")):
        sub = diff[diff.check.str.startswith(cid)]
        if len(sub):
            nbfig.limits(sub, f"{cid}: wake-code-place-q4 minus the reference, against the limit", f"difference ({unit})",
                   label="point").update_layout(height=380).show()
diff